# Inter-Coder Reliability (ICR) — Krippendorff's Alpha
## Annotator vs Ground Truth (2-annotator design)

**Design:** each sentence was coded by **one human annotator** and the **ground truth** (aka Isabelle and me)
| Measure | What it captures |
|---|---|
| **α (MASI)** | Multi-label agreement — penalises partial label-set mismatches using MASI distance |
| **α (nominal / binary)** | Binary mention-detection agreement — does the annotator find *any* group at all? |

**Outputs:**  `icr_by_outlet.csv` · `icr_by_annotator.csv` · `icr_by_category.csv` · `icr_by_label.csv`

---


In [1]:
!pip install jinja2 -q

## 1 · Imports

In [2]:
import ast, re, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import krippendorff     
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", "{:.4f}".format)
print("Libraries loaded ✓")


Libraries loaded ✓


## 2 · File Paths

Point these two variables at your input files.


In [ ]:
GT_FILE  = "../annotations_ground_truth.xlsx"
ANN_FILE = "../annotations_by_coder.xlsx"

## 3 · Codebook

9 broad categories with their canonical labels.  
Edit here if the codebook changes.


In [4]:
CODEBOOK: dict[str, list[str]] = {
    "Socio-economic position": [
        "lower class", "middle class", "upper class",
        "capital owners, investors and shareholders",
        "unskilled or unqualified", "skilled or qualified",
    ],
    "Labor market position": [
        "wage and salary earners", "civil servants", "CEOs and corporate leaders",
        "employers", "entrepreneurs", "self-employed and freelancers",
        "unemployed", "retirees", "housewives and househusbands",
    ],
    "Age and family status": [
        "parents and families", "minors", "youth",
        "middle-aged and pre-retirement age groups", "elderly", "couples", "singles",
    ],
    "Identities and minority/majority status": [
        "men", "women", "cisgender and heterosexuals", "lgbtqia+", "disabled people",
        "people with an immigration background, including immigrants",
        "ethnic and racial minorities",
        "christians", "jews", "muslims",
        "multiple (or other) religious or minority groups",
    ],
    "Profession": [
        "athletes", "authors and artists", "doctors", "farmers and fishermen",
        "health and care professionals", "journalists", "legal professionals",
        "politicians and high-ranking officials", "sex workers",
        "scientists and professors", "security forces", "soldiers",
        "teachers and educators", "other professions",
    ],
    "Social roles and behavior": [
        "consumers and clients", "car drivers", "patients",
    ],
    "Social deviance": [
        "extremists",
        "terrorists, rebels, revolutionaries and/or movements of armed resistance",
        "offenders, criminals, prisoners and/or accused people",
        "drug addicts",
    ],
    "Real estate ownership": [
        "real-estate owners", "tenants", "homeless",
    ],
    "Others": ["others"],
}

CANONICAL    = {lbl for lbls in CODEBOOK.values() for lbl in lbls}
LABEL_TO_CAT = {lbl: cat for cat, lbls in CODEBOOK.items() for lbl in lbls}

print(f"Codebook loaded: {len(CODEBOOK)} categories · {len(CANONICAL)} canonical labels")


Codebook loaded: 9 categories · 58 canonical labels


## 4 · Label Normalisation

Maps old/variant annotation labels to canonical codebook labels.


In [5]:
# Labels that contain commas internally — must be shielded before comma-splitting
COMMA_LABELS = [
    "offenders, criminals, prisoners and/or accused people",
    "terrorists, rebels, revolutionaries and/or movements of armed resistance",
    "capital owners, investors and shareholders",
    "multiple (or other) religious or minority groups",
]

NORMALISE: dict[str, str] = {
    # ── Socio-economic ────────────────────────────────────────────────────
    "poor": "lower class", "underprivileged": "lower class",
    "unskilled and/or underprivileged": "unskilled or unqualified",
    "qualified": "skilled or qualified", "qualified and graduates": "skilled or qualified",
    "investors and stakeholders": "capital owners, investors and shareholders",
    # ── Labor market ──────────────────────────────────────────────────────
    "employees": "wage and salary earners", "precarious employees": "wage and salary earners",
    "working active population": "wage and salary earners",
    "housewife and househusband": "housewives and househusbands",
    "self-employed/freelancers": "self-employed and freelancers",
    "leaders": "CEOs and corporate leaders", "ceos and corporate leaders": "CEOs and corporate leaders",
    "enterprises": "entrepreneurs", "large enterprises": "entrepreneurs",
    "small- and middle-size enterprises": "entrepreneurs", "specific sector": "entrepreneurs",
    "entrepreneurs (smes)": "entrepreneurs", "entrepreneurs (large enterprises)": "entrepreneurs",
    "entrepreneurs in [specific] sector": "entrepreneurs",
    # ── Age & family ──────────────────────────────────────────────────────
    "minors, including children and pupils": "minors",
    "youth, including students and apprentices": "youth",
    "middle-aged": "middle-aged and pre-retirement age groups",
    "older age group": "elderly", "seniors": "elderly",
    # ── Identities ────────────────────────────────────────────────────────
    "immigrants": "people with an immigration background, including immigrants",
    "visible and ethnic minorities": "ethnic and racial minorities",
    "ethnic minorities": "ethnic and racial minorities", "minorities": "ethnic and racial minorities",
    "east germans": "ethnic and racial minorities", "west germans": "ethnic and racial minorities",
    "expatriates": "ethnic and racial minorities",
    "white": "ethnic and racial minorities", "white people": "ethnic and racial minorities",
    "ethnic germans": "ethnic and racial minorities",                        # ← NEW
    "visible minorities": "ethnic and racial minorities",                   # ← NEW
    "language and ethnic minorities": "ethnic and racial minorities",       # ← NEW
    "lgbtqi*": "lgbtqia+", "lgbtqqia+": "lgbtqia+",
    "cis & heterosexuals": "cisgender and heterosexuals",
    "religious groups": "multiple (or other) religious or minority groups",
    "territorial language minorities": "multiple (or other) religious or minority groups",
    "religious minorities": "multiple (or other) religious or minority groups",  # ← NEW
    "disabled": "disabled people",
    # ── Profession ────────────────────────────────────────────────────────
    "other profession": "other professions", "scientists": "scientists and professors",
    "prostitutes": "sex workers", "social professions": "other professions",
    "engineers": "other professions", "lobbyists": "other professions",
    "people working in the public sector": "civil servants",
    "hunters": "other professions",                                         # ← NEW
    # ── Social roles ──────────────────────────────────────────────────────
    "commuters": "consumers and clients", "cyclists": "car drivers", "pedestrians": "car drivers",
    "road carriers": "consumers and clients", "air travellers": "consumers and clients",
    "users of certain transportation modes": "consumers and clients",
    "public transport passengers": "consumers and clients", "insured persons": "patients",
    "consumers": "consumers and clients",                                   # ← NEW
    "tax payers": "others", "gun owners": "others",
    # ── Social deviance ───────────────────────────────────────────────────
    "tax evaders and white collar criminals": "offenders, criminals, prisoners and/or accused people",
    "offenders or criminals": "offenders, criminals, prisoners and/or accused people",  # ← NEW
    "terrorists": "terrorists, rebels, revolutionaries and/or movements of armed resistance",  # ← NEW
    # ── Real estate ───────────────────────────────────────────────────────
    "home owner": "real-estate owners", "land owner": "real-estate owners",
    "landlords": "real-estate owners", "real-estate owner": "real-estate owners",
    "real estate owners": "real-estate owners",
    # ── Geography → Others ────────────────────────────────────────────────
    "inhabitants of cities": "others", "inhabitants of rural or underserved areas": "others",
    "inhabitants of other areas": "others", "inhabitants of overseas": "others",
    "inhabitants of specific sites": "others", "inhabitants of underprivileged areas": "others",
    # ── Residual ──────────────────────────────────────────────────────────
    "victims of crimes": "others", "victims of state violence": "others",
    "victims of german history": "others", "whistle-blower and witnesses": "others",
    "volunteers": "others", "people without public social protection": "others",
    "heirs": "others",
    "other": "others",                                                      # ← NEW (Mathieu's 'Other')
}


def normalise_label(raw: str) -> str | None:
    raw = raw.strip().lower()
    if raw in ("target abroad", "nan", ""):
        return None          # not a social-group label — exclude
    if raw in NORMALISE:
        return NORMALISE[raw]
    if raw in CANONICAL:
        return raw
    for c in CANONICAL:     # case-insensitive fallback
        if raw == c.lower():
            return c
    return "others"          # unknown → catch-all


print("Normalisation map loaded ✓")
print(f"  {len(NORMALISE)} explicit mappings  |  unknowns → 'others'")

Normalisation map loaded ✓
  85 explicit mappings  |  unknowns → 'others'


## 5 · Label Parsing

Two parsers for the two different label formats in the input files.


In [6]:
def _protect(t: str) -> str:
    for cl in sorted(COMMA_LABELS, key=len, reverse=True):
        t = t.replace(cl, cl.replace(", ", "|||"))
    return t

def _unprotect(t: str) -> str:
    return t.replace("|||", ", ")


def parse_ann_label(val) -> frozenset[str]:
    """
    Annotation file label field → normalised frozenset.
    Handles:  NaN · 'others' · 'others, target abroad' · 'athletes; athletes'
              'offenders, criminals, ...' · 'politicians ...; others'
    """
    if pd.isna(val):
        return frozenset()
    val = str(val).strip()
    if val in ("nan", ""):
        return frozenset()
    protected = _protect(val.lower())
    labels = set()
    for chunk in re.split(r";\s*", protected):      
        chunk = _unprotect(chunk).strip()
        if not chunk or chunk == "nan":
            continue
        for piece in _protect(chunk).split(", "):    
            piece = _unprotect(piece).strip()
            if piece:
                canon = normalise_label(piece)
                if canon is not None:
                    labels.add(canon)
    return frozenset(labels)


def parse_gt_label(val) -> frozenset[str]:
    """
    Ground-truth column  [[start, end, 'label'], ...]  → normalised frozenset.
    """
    if pd.isna(val):
        return frozenset()
    try:
        labels = set()
        for s in ast.literal_eval(str(val)):
            if isinstance(s, list) and len(s) >= 3:
                canon = normalise_label(str(s[2]))
                if canon is not None:
                    labels.add(canon)
        return frozenset(labels)
    except Exception:
        return frozenset()


# ── Sanity checks ─────────────────────────────────────────────────────────────
_tests = [
    ("others, target abroad",          frozenset({"others"})),
    ("offenders, criminals, prisoners and/or accused people, target abroad",
     frozenset({"offenders, criminals, prisoners and/or accused people"})),
    ("athletes; athletes",             frozenset({"athletes"})),
    ("politicians and high-ranking officials; others",
     frozenset({"politicians and high-ranking officials", "others"})),
]
assert all(parse_ann_label(i) == e for i, e in _tests), "parse_ann_label failed sanity check!"
print("parse_ann_label  ✓")
print("parse_gt_label   ✓")


parse_ann_label  ✓
parse_gt_label   ✓


## 6 · MASI Distance & Krippendorff's Alpha


In [7]:
def masi_distance(A: frozenset, B: frozenset) -> float:
    """MASI distance between two label sets. Range [0, 1]."""
    if A == B:
        return 0.0
    inter = len(A & B)
    union = len(A | B)
    jaccard = inter / union if union > 0 else 0.0
    m = 0.67 if (A <= B or B <= A) else (0.33 if inter > 0 else 0.0)
    return 1.0 - jaccard * m


def kripp_alpha_masi(pairs: list[tuple[frozenset, frozenset]]) -> float:
    """
    Krippendorff's alpha with MASI distance for 2-annotator multi-label data.

    Formula (Krippendorff 2004):
        α = 1 - D_o / D_e
    where
        D_o = mean pairwise distance of observed coincidences
        D_e = mean pairwise distance expected by chance

    Args:
        pairs: list of (annotator_labelset, groundtruth_labelset) per sentence.

    Returns:
        float alpha, or np.nan if insufficient data.
    """
    if not pairs:
        return np.nan

    # Observed disagreement
    Do_sum = sum(masi_distance(p, g) for p, g in pairs)
    D_o    = Do_sum / len(pairs)

    # Expected disagreement — all values pooled
    all_values = [ls for pair in pairs for ls in pair]
    n = len(all_values)
    if n < 2:
        return np.nan
    De_sum = sum(
        masi_distance(all_values[i], all_values[j])
        for i in range(n) for j in range(n) if i != j
    )
    D_e = De_sum / (n * (n - 1))

    return np.nan if D_e == 0 else round(1.0 - D_o / D_e, 6)


def kripp_alpha_binary(pairs: list[tuple[frozenset, frozenset]]) -> float:
    """
    Krippendorff's alpha (nominal) for binary mention-detection:
    1 = sentence contains ≥1 group mention, 0 = no mention.
    Uses the krippendorff library for the 2-row reliability matrix.
    """
    rel = np.array(
        [(1.0 if p else 0.0, 1.0 if g else 0.0) for p, g in pairs],
        dtype=float
    ).T   # shape (2, N) — 2 annotators × N sentences
    try:
        return round(float(krippendorff.alpha(
            reliability_data=rel, level_of_measurement="nominal"
        )), 6)
    except Exception:
        return np.nan


# ── Validation ────────────────────────────────────────────────────────────────
_p1 = [(frozenset({"athletes"}), frozenset({"athletes"})),
       (frozenset({"women"}),    frozenset({"women"}))]
assert kripp_alpha_masi(_p1) == 1.0, "Perfect agreement should give alpha=1.0"

_p2 = [(frozenset({"athletes"}), frozenset({"women"})),
       (frozenset({"women"}),    frozenset({"athletes"}))]
print(f"Validation — perfect agreement : α = {kripp_alpha_masi(_p1):.4f}  (expected 1.0)")
print(f"Validation — mirrored mismatch : α = {kripp_alpha_masi(_p2):.4f}  (expected ≤ 0)")
print("MASI alpha functions  ✓")
print("Binary alpha function ✓")


Validation — perfect agreement : α = 1.0000  (expected 1.0)
Validation — mirrored mismatch : α = -0.5000  (expected ≤ 0)
MASI alpha functions  ✓
Binary alpha function ✓


## 7 · Load & Merge Data

In [9]:
gt  = pd.read_excel(GT_FILE)
ann = pd.read_excel(ANN_FILE)

# Coerce outlet to string and drop the 2 rows with no outlet
gt["outlet"] = gt["outlet"].astype(str)
gt = gt[gt["outlet"] != "nan"].copy()

gt["gt_labels"]   = gt["ground_truth"].apply(parse_gt_label)
ann["ann_labels"] = ann["label"].apply(parse_ann_label)

merged = ann.merge(
    gt[["text", "outlet", "country", "gt_labels"]],
    on="text", how="inner"
)

# null-safe sorted unique helper
def sorted_unique(series):
    return sorted(series.dropna().astype(str).unique())

print(f"Ground truth   : {len(gt):,} sentences · outlets: {sorted_unique(gt['outlet'])}")
print(f"Annotations    : {len(ann):,} rows · annotators: {sorted_unique(ann['annotator'])}")
print(f"Merged         : {len(merged):,} rows ({merged['text'].nunique():,} unique sentences)")
print()

# Coverage report — multiple annotators per sentence is fine (each compared vs GT independently)
max_ann = merged.groupby("text")["annotator"].nunique().max()
mean_ann = merged.groupby("text")["annotator"].nunique().mean()
multi = (merged.groupby("text")["annotator"].nunique() > 1).sum()
print(f"Max annotators per sentence : {max_ann}")
print(f"Mean annotators per sentence: {mean_ann:.2f}")
print(f"Sentences with >1 annotator : {multi:,}  (each still compared vs GT independently)")
print()

display(merged.groupby(["outlet","annotator"])["text"]
        .nunique().unstack(fill_value=0))

Ground truth   : 9,639 sentences · outlets: ['Bild', 'FAZ', 'Figaro', 'Liberation', 'Mediapart', 'Monde', 'MondeDiplo', 'Parisien', 'SZ', 'Spiegel', 'Welt', 'Zeit']
Annotations    : 9,119 rows · annotators: ['Celine', 'Elisa', 'Isabelle', 'Mathieu', 'Selma_DE', 'Selma_FR', 'Stella', 'Theres']
Merged         : 9,120 rows (8,939 unique sentences)

Max annotators per sentence : 2
Mean annotators per sentence: 1.02
Sentences with >1 annotator : 175  (each still compared vs GT independently)



annotator,Celine,Elisa,Isabelle,Mathieu,Selma_DE,Selma_FR,Stella,Theres
outlet,,,,,,,,
Bild,291,137,0,0,88,0,286,126
FAZ,306,134,0,0,76,0,302,131
Figaro,0,0,289,234,0,343,0,0
Liberation,0,0,0,226,0,0,0,0
Mediapart,0,0,0,364,0,0,0,0
Monde,0,0,242,264,0,347,0,0
MondeDiplo,0,0,0,374,0,0,0,0
Parisien,0,0,247,261,0,370,0,0
SZ,272,138,0,0,91,0,291,129


## 8 · Overall ICR

In [10]:
pairs_all = list(zip(merged["ann_labels"], merged["gt_labels"]))

alpha_masi_overall   = kripp_alpha_masi(pairs_all)
alpha_binary_overall = kripp_alpha_binary(pairs_all)

print("Overall inter-coder reliability (annotator vs ground truth)")
print(f"  N sentences     : {len(pairs_all):,}")
print(f"  α  MASI         : {alpha_masi_overall:.4f}   ← main ICR measure")
print(f"  α  binary       : {alpha_binary_overall:.4f}   ← mention-detection only")


Overall inter-coder reliability (annotator vs ground truth)
  N sentences     : 9,120
  α  MASI         : 0.8475   ← main ICR measure
  α  binary       : 0.8080   ← mention-detection only


## 9 · ICR by Outlet — Table 3


In [ ]:
outlet_rows = []
for outlet in sorted(merged["outlet"].dropna().unique()):
    sub   = merged[merged["outlet"] == outlet]
    pairs = list(zip(sub["ann_labels"], sub["gt_labels"]))
    outlet_rows.append({
        "Outlet":       outlet,
        "Country":      sub["country"].iloc[0],
        "α MASI":       kripp_alpha_masi(pairs),
        "α binary":     kripp_alpha_binary(pairs),
        "# Sentences":  len(pairs),
        "# Annotators": sub["annotator"].nunique(),
        "Annotators":   ", ".join(sorted(sub["annotator"].unique())),
    })

df_outlets = pd.DataFrame(outlet_rows)

print("=" * 70)
print("TABLE 3 — Inter-coder reliability at the sentence level")
print("=" * 70)
for country_name, code in [("German Newspapers", "Germany"), ("French Newspapers", "France")]:
    print(f"\n{country_name}")
    sub = df_outlets[df_outlets["Country"] == code].sort_values("Outlet")
    for _, r in sub.iterrows():
        flag = "✓" if r["α MASI"] >= 0.80 else ("~" if r["α MASI"] >= 0.67 else "✗")
        print(f"  {r['Outlet']:<20} α={r['α MASI']:.2f} {flag}   "
              f"N={r['# Sentences']:,}   annotators={r['# Annotators']}")
print(f"\nOverall  α={alpha_masi_overall:.2f}")

display(df_outlets[["Outlet","Country","α MASI","α binary","# Sentences","# Annotators","Annotators"]]
        .set_index("Outlet"))

df_outlets.to_csv("icr_by_outlet.csv", index=False)
print("\n→ saved: icr_by_outlet.csv")

## 10 · ICR by Annotator

In [ ]:
ann_rows = []
for annotator in sorted(merged["annotator"].unique()):
    sub   = merged[merged["annotator"] == annotator]
    pairs = list(zip(sub["ann_labels"], sub["gt_labels"]))
    ann_rows.append({
        "Annotator": annotator,
        "α MASI":    kripp_alpha_masi(pairs),
        "α binary":  kripp_alpha_binary(pairs),
        "N":         len(pairs),
    })

df_ann = pd.DataFrame(ann_rows).set_index("Annotator")
display(df_ann)

df_ann.reset_index().to_csv("icr_by_annotator.csv", index=False)
print("→ saved: icr_by_annotator.csv")

## 11 · ICR by Broad Category

For each category, both label sets are restricted to that category's labels.
Only sentences where at least one of the two restricted sets is non-empty are included.


In [ ]:
cat_rows = []
for cat, cat_labels in CODEBOOK.items():
    cat_set = frozenset(cat_labels)
    cat_pairs = [
        (pred & cat_set, gold & cat_set)
        for pred, gold in pairs_all
        if (pred & cat_set) or (gold & cat_set)
    ]
    if not cat_pairs:
        continue
    cat_rows.append({
        "Category": cat,
        "α MASI":   kripp_alpha_masi(cat_pairs),
        "α binary": kripp_alpha_binary(cat_pairs),
        "N (sentences with label)": len(cat_pairs),
    })

df_cats = (pd.DataFrame(cat_rows)
             .sort_values("α MASI", ascending=False)
             .set_index("Category"))
display(df_cats)

df_cats.reset_index().to_csv("icr_by_category.csv", index=False)
print("→ saved: icr_by_category.csv")

## 12 · ICR by Specific Label

Binary α per label: did annotator and ground truth agree on whether this
specific label is present in the sentence?  
Labels with fewer than 5 co-annotated sentences are excluded.


In [ ]:
all_labels_in_data = set()
for pred, gold in pairs_all:
    all_labels_in_data |= pred | gold

label_rows = []
for lbl in sorted(all_labels_in_data):
    lbl_pairs = [
        (frozenset([lbl]) if lbl in pred else frozenset(),
         frozenset([lbl]) if lbl in gold else frozenset())
        for pred, gold in pairs_all
        if lbl in pred or lbl in gold
    ]
    if len(lbl_pairs) < 5:
        continue
    label_rows.append({
        "Label":    lbl,
        "Category": LABEL_TO_CAT.get(lbl, "Others"),
        "α MASI":   kripp_alpha_masi(lbl_pairs),
        "α binary": kripp_alpha_binary(lbl_pairs),
        "N (sentences with label)": len(lbl_pairs),
    })

df_labels = (pd.DataFrame(label_rows)
               .sort_values("α MASI", ascending=False)
               .set_index("Label"))
display(df_labels)

df_labels.reset_index().to_csv("icr_by_label.csv", index=False)
print("→ saved: icr_by_label.csv")

## 13 · Paper-Ready Table 3

In [20]:
print("Table 3. Inter-coder reliability at the sentence level.")
print(f"{'Newspaper':<28} {'α (MASI)':>10}  {'# sentences':>13}  {'# annotators':>14}")
print("-" * 72)

for label, code in [("German Newspapers", "Germany"), ("French Newspapers", "France")]:
    print(label)
    sub = df_outlets[df_outlets["Country"] == code].sort_values("Outlet")
    for _, r in sub.iterrows():
        print(f"  {r['Outlet']:<26} {r['α MASI']:>10.2f}  "
              f"{r['# Sentences']:>13,}  {r['# Annotators']:>14}")

print("-" * 72)
print(f"  {'Overall':<26} {alpha_masi_overall:>10.2f}  {len(pairs_all):>13,}")


Table 3. Inter-coder reliability at the sentence level.
Newspaper                      α (MASI)    # sentences    # annotators
------------------------------------------------------------------------
German Newspapers
  Bild                             0.81            930               5
  FAZ                              0.84            950               5
  SZ                               0.82            922               5
  Spiegel                          0.83            906               5
  Welt                             0.84            914               5
  Zeit                             0.81            936               5
French Newspapers
  Figaro                           0.95            866               3
  Liberation                       0.74            226               1
  Mediapart                        0.69            364               1
  Monde                            0.94            853               3
  MondeDiplo                       0.68            374